# YOLO11s tuning experiments on the VisDrone-DET pipeline

**Question.** Which single change to the fine-tuning recipe of notebook 10 improves the
clean-fold mAP? We also ask which inference settings help the best checkpoint. The
reference is the submitted checkpoint of notebook 10: one fine-tuning epoch, clean-fold
mAP 0.3553, Kaggle Public 0.34389.

**Evidence behind the experiments.**

- **Fine-tuning erodes the VisDrone prior.** In notebook 10, native validation peaked
  after the first epoch, still inside the learning-rate warmup, and declined while
  training loss fell. That suggests forgetting of the VisDrone-DET scene diversity.
  Two experiments reduce how far fine-tuning moves the weights: a four times lower
  learning rate, and a frozen backbone.
- **Class confusion.** Confident predictions misclassify 24% to 29% of van, truck and
  bicycle objects. A larger classification loss gain is our loss modification.
- **Missed small objects.** 37% of small objects are missed. Every second frame doubles
  the distinct training views of the same clips. This is the data experiment.
- **Inference strategy.** A larger input, test-time augmentation and a higher confidence
  threshold are evaluated without retraining. This separates inference effects from
  model effects.

**Protocol.** Each training experiment changes one setting of `yolo11s_native_1280.toml`.
Every run starts from the same converted VisDrone-DET checkpoint and uses the same 57
training clips and the same clean six-clip validation fold. Each trains for at most 8
epochs and stops after 3 epochs without improvement. The shorter schedule is shared by all
experiments and does not change the first epoch: Ultralytics warms up towards the initial
learning rate during epoch 1 regardless of the epoch count, and mosaic is closed only for
the last 5 epochs. The submitted checkpoint, which is from epoch 1, is therefore a valid
reference. `best.pt` and `last.pt` are compared on the clean fold with COCO mAP at 100
detections, and the better one exports a test submission. The experiments are listed in
`config/yolo11s_native_experiments.toml`.

**Running on Kaggle.** Use the same setup as notebook 10: competition data, the
`hm2-yolo-native` dataset, GPU T4 x2 and the `HM2_DRIVE_TOKEN` secret. With a single GPU
the notebook keeps 8 images per GPU and the same effective batch, but training takes
about twice as long. Set
`SHOULD_TRAIN = True` and use *Save Version → Save & Run All*. With the measured
throughput the four runs take at most about 3.5 hours of training. Evaluation, export and
data preparation add about one hour, and early stopping usually shortens the runs.
Each run is mirrored to Drive, so a rerun skips finished experiments and resumes an
interrupted one.

## 1. Environment and configuration

The first cell is the bootstrap of notebook 10. It installs pinned packages and loads
the project code from the attached dataset or from Drive.

In [ ]:
import hashlib
import importlib
import importlib.metadata
import importlib.util
import json
import logging
import os
import platform
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from zipfile import ZipFile

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()
BUNDLE_NAME = "yolo_native_project.zip"
DRIVE_ROOT = "16wH9gNp13kfa4T039nsiFcZwYOlEJI5Z"
SHOULD_USE_DRIVE = IS_KAGGLE
PINNED_PACKAGES = {
    "ultralytics": "ultralytics==8.3.235",
    "thop": "ultralytics-thop>=2.0.18",
    "faster_coco_eval": "faster-coco-eval>=1.8.0",
}
REQUIREMENTS = {
    "polars": "polars>=1.31",
    "plotly": "plotly>=6",
    "pycocotools": "pycocotools>=2.0.11",
    "googleapiclient": "google-api-python-client",
    "google.oauth2": "google-auth",
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
LOGGER = logging.getLogger("yolo_native_pipeline")

attached_bundles, wheels = [], None
if IS_KAGGLE:
    attached_bundles = sorted(
        {path for path in KAGGLE_INPUT.rglob(BUNDLE_NAME) if path.is_file()}
        | {path.parent for path in KAGGLE_INPUT.rglob("bundle_manifest.json")}
    )
    if len(attached_bundles) > 1:
        raise ValueError(f"Attach exactly one code bundle: {attached_bundles}")
    if attached_bundles:
        bundle_dataset = KAGGLE_INPUT / attached_bundles[0].relative_to(KAGGLE_INPUT).parts[0]
        wheels = next((path.parent for path in bundle_dataset.rglob("*.whl")), None)
    pinned = [
        package
        for module, package in PINNED_PACKAGES.items()
        if importlib.util.find_spec(module) is None
    ]
    if (
        "ultralytics==8.3.235" not in pinned
        and importlib.metadata.version("ultralytics") != "8.3.235"
    ):
        pinned.append("ultralytics==8.3.235")
    missing = [
        package
        for module, package in REQUIREMENTS.items()
        if importlib.util.find_spec(module) is None
    ]
    installer_prefix = []
    if wheels is not None:
        installer_prefix = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--find-links",
            str(wheels),
        ]
    elif pinned or missing:
        uv_path = shutil.which("uv")
        if uv_path is None:
            installer = Path("/kaggle/working/install_uv.sh")
            urllib.request.urlretrieve("https://astral.sh/uv/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
            uv_path = str(Path.home() / ".local/bin/uv")
        installer_prefix = [uv_path, "pip", "install", "--system", "--python", sys.executable]
    if pinned:
        subprocess.run([*installer_prefix, "--no-deps", *pinned], check=True)
    if missing:
        subprocess.run([*installer_prefix, *missing], check=True)
    LOGGER.info(
        "Installed packages without dependencies: %s; with dependencies: %s", pinned, missing
    )

drive_service = None
if SHOULD_USE_DRIVE:
    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build

    UserSecretsClient = importlib.import_module("kaggle_secrets").UserSecretsClient
    token = json.loads(UserSecretsClient().get_secret("HM2_DRIVE_TOKEN"))
    if not token.get("refresh_token"):
        raise ValueError("HM2_DRIVE_TOKEN must contain refreshable Drive credentials.")
    drive_service = build(
        "drive",
        "v3",
        credentials=Credentials.from_authorized_user_info(token),
        cache_discovery=False,
    )
    del token

if IS_KAGGLE:
    REPOSITORY_ROOT = Path("/kaggle/temp/hm2_yolo_code")
    if attached_bundles:
        archive_path = attached_bundles[0]
    elif drive_service is not None:
        from googleapiclient.http import MediaIoBaseDownload

        project = {"id": DRIVE_ROOT}
        for name in ("inputs", BUNDLE_NAME):
            response = (
                drive_service.files()
                .list(
                    q=f"'{project['id']}' in parents and trashed = false",
                    fields="files(id,name,size,md5Checksum)",
                    pageSize=1000,
                )
                .execute(num_retries=3)
            )
            matches = [entry for entry in response["files"] if entry["name"] == name]
            if len(matches) != 1:
                raise FileNotFoundError(f"Expected exactly one {name} in the Drive inputs path.")
            project = matches[0]
        archive_path = Path("/kaggle/temp") / BUNDLE_NAME
        archive_path.parent.mkdir(parents=True, exist_ok=True)
        incoming = archive_path.with_suffix(".incoming")
        with incoming.open("wb") as stream:
            transfer = MediaIoBaseDownload(
                stream,
                drive_service.files().get_media(fileId=project["id"]),
                chunksize=4 * 1024**2,
            )
            has_completed = False
            while not has_completed:
                _, has_completed = transfer.next_chunk(num_retries=3)
        with incoming.open("rb") as stream:
            if hashlib.file_digest(stream, "md5").hexdigest() != project["md5Checksum"]:
                raise ValueError("Code bundle checksum mismatch after the Drive download.")
        incoming.replace(archive_path)
    else:
        raise FileNotFoundError(
            f"Attach a dataset containing {BUNDLE_NAME} or set SHOULD_USE_DRIVE with HM2_DRIVE_TOKEN."
        )
    shutil.rmtree(REPOSITORY_ROOT, ignore_errors=True)
    if archive_path.is_dir():
        shutil.copytree(archive_path, REPOSITORY_ROOT)
    else:
        REPOSITORY_ROOT.mkdir(parents=True)
        with ZipFile(archive_path) as archive:
            for entry in archive.infolist():
                if not (REPOSITORY_ROOT / entry.filename).resolve().is_relative_to(REPOSITORY_ROOT):
                    raise ValueError(f"Unsafe path in the code bundle: {entry.filename}")
            archive.extractall(REPOSITORY_ROOT)
    LOGGER.info("Using project code from %s", archive_path)
else:
    working_directory = Path.cwd().resolve()
    REPOSITORY_ROOT = next(
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "pyproject.toml").is_file() and (directory / "src/hm2").is_dir()
    )
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

In [ ]:
import tomllib
from io import BytesIO

import numpy as np
import plotly.express as px
import polars as pl
import torch
import ultralytics
from IPython.display import Markdown, display
from ultralytics import YOLO

from hm2.data import CLASS_NAMES, find_dataset_root, load_tables
from hm2.experiment_checkpoint import publish_snapshot, write_json
from hm2.faster_rcnn_experiments import find_competition_root
from hm2.yolo_data import prepare_yolo_dataset, select_dataset_frames, verify_yolo_dataset
from hm2.yolo_pipeline import (
    CheckpointMirror,
    apply_experiment,
    build_yolo_submission,
    evaluate_checkpoints,
    load_pipeline_settings,
    predict_frames,
    restore_run_from_mirror,
    training_arguments,
    write_coco_subset,
)

logging.getLogger("faster_coco_eval").setLevel(logging.WARNING)
ultralytics.settings.update(
    {
        name: False
        for name in (
            "clearml",
            "comet",
            "dvc",
            "hub",
            "mlflow",
            "neptune",
            "raytune",
            "tensorboard",
            "wandb",
        )
    }
)
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)

SHOULD_PREPARE_DATA = True
SHOULD_TRAIN = False
SHOULD_EVALUATE = True
SHOULD_EXPORT_SUBMISSIONS = True

PLAN = tomllib.loads((REPOSITORY_ROOT / "config/yolo11s_native_experiments.toml").read_text())
BASE = load_pipeline_settings(REPOSITORY_ROOT / "config" / PLAN["base_config"])
EXPERIMENTS = PLAN["experiments"]
INFERENCE_EXPERIMENTS = PLAN["inference_experiments"]
BATCH_ID = "yolo11s_native_tuning_seed2026"
SECONDS_PER_TRAIN_IMAGE = 0.0457
VALIDATION_SECONDS = 60.4
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else REPOSITORY_ROOT
TEMP_ROOT = Path("/kaggle/temp") if IS_KAGGLE else REPOSITORY_ROOT / "data"
WEIGHTS_DIRECTORY = REPOSITORY_ROOT / ("weights" if IS_KAGGLE else "data/weights")
INITIAL_WEIGHTS = WEIGHTS_DIRECTORY / BASE["model"]["model_id"]
BASELINE_WEIGHTS = WEIGHTS_DIRECTORY / PLAN["baseline_checkpoint"]
RUN_PROJECT = WORK_ROOT / "runs" / "yolo_native" / BATCH_ID
MIRROR_ROOT = TEMP_ROOT / "drive_mirror" / BATCH_ID
SUBMISSION_DIRECTORY = WORK_ROOT / "submissions" if IS_KAGGLE else RUN_PROJECT / "submissions"
for directory in (RUN_PROJECT, SUBMISSION_DIRECTORY, TEMP_ROOT / "prepared"):
    directory.mkdir(parents=True, exist_ok=True)
if SHOULD_TRAIN and not torch.cuda.is_available():
    raise RuntimeError("Training requires a CUDA runtime. Select a GPU accelerator.")
device = 0 if torch.cuda.is_available() else "cpu"
GPU_COUNT = torch.cuda.device_count()
for index in range(GPU_COUNT):
    try:
        torch.zeros(1, device=f"cuda:{index}").add_(1).item()
    except RuntimeError as error:
        raise RuntimeError(
            f"{torch.cuda.get_device_name(index)} cannot run this PyTorch build. Select GPU T4 x2."
        ) from error
LOGGER.info("GPUs: %s", [torch.cuda.get_device_name(index) for index in range(GPU_COUNT)])

# The recipe uses DDP on two GPUs with 8 images per GPU. With fewer GPUs we keep 8 images
# per GPU. Ultralytics accumulates gradients to its nominal batch of 64 and scales weight
# decay by the same rule, so every optimizer step still sees 64 images; epochs take longer.
configured_gpus = len(str(BASE["training"]["device"]).split(","))
gpu_slowdown = 1.0
if 0 < GPU_COUNT < configured_gpus:
    images_per_gpu = BASE["training"]["batch"] // configured_gpus
    BASE["training"].update(
        device=",".join(str(index) for index in range(GPU_COUNT)),
        batch=images_per_gpu * GPU_COUNT,
    )
    gpu_slowdown = configured_gpus / GPU_COUNT
    LOGGER.warning(
        "Found %d GPU(s) instead of %d: training on device %s with batch %d.",
        GPU_COUNT,
        configured_gpus,
        BASE["training"]["device"],
        BASE["training"]["batch"],
    )

plan_rows = []
for name, overrides in EXPERIMENTS.items():
    settings = apply_experiment(BASE, overrides)
    epochs = settings["training"]["max_epochs"]
    frames_per_epoch = settings["dataset"]["expected_train_frames"]
    plan_rows.append(
        {
            "Experiment": name,
            "Change": overrides["description"],
            "Maximum epochs": epochs,
            "Patience": settings["training"]["patience"],
            "Training frames": frames_per_epoch,
            "Device": settings["training"]["device"],
            "Batch": settings["training"]["batch"],
            "Projected hours, upper bound": epochs
            * (frames_per_epoch * SECONDS_PER_TRAIN_IMAGE * gpu_slowdown + VALIDATION_SECONDS)
            / 3600,
        }
    )
LOGGER.info(
    "Switches: prepare=%s, train=%s, evaluate=%s, export=%s, drive=%s",
    SHOULD_PREPARE_DATA,
    SHOULD_TRAIN,
    SHOULD_EVALUATE,
    SHOULD_EXPORT_SUBMISSIONS,
    drive_service is not None,
)
pl.from_dicts(plan_rows)

## 2. Data preparation and the clean validation fold

Each distinct frame stride gets its own prepared dataset, built with the same resize and
ignored-region masking as notebook 10 and checked before training. All experiments
share the clean validation fold and its merged COCO annotations.

In [ ]:
found_root = (
    find_competition_root(KAGGLE_INPUT)
    if IS_KAGGLE
    else find_dataset_root(REPOSITORY_ROOT / "data")
)
if found_root is None:
    raise FileNotFoundError("Attach the competition data on Kaggle or extract it under data/.")
SOURCE_ROOT: Path = found_root
frames, objects, ignored = load_tables(SOURCE_ROOT)
test_frames = pl.read_csv(SOURCE_ROOT / "test_images.csv", schema_overrides={"image_id": pl.String})
dataset_by_stride = {
    apply_experiment(BASE, overrides)["dataset"]["frame_stride"]: apply_experiment(BASE, overrides)[
        "dataset"
    ]
    for overrides in EXPERIMENTS.values()
}
DATA_YAMLS, preparation_rows = {}, []
for stride, dataset_settings in sorted(dataset_by_stride.items()):
    dataset_root = TEMP_ROOT / "prepared" / f"yolo11s_native_1280_stride{stride}"
    manifest_path = RUN_PROJECT / f"dataset_manifest_stride{stride}.json"
    if SHOULD_PREPARE_DATA:
        manifest = prepare_yolo_dataset(
            SOURCE_ROOT,
            dataset_root,
            frames,
            objects,
            ignored,
            dataset_settings,
            os.cpu_count() or 4,
        )
        write_json(manifest_path, manifest)
    else:
        manifest = json.loads(manifest_path.read_text())
    checks = verify_yolo_dataset(dataset_root, frames, objects, dataset_settings, manifest)
    if not checks["is_pass"].all():
        raise ValueError(f"Prepared dataset checks failed:\n{checks.filter(~pl.col('is_pass'))}")
    DATA_YAMLS[stride] = Path(manifest["data_yaml"])
    preparation_rows.append(
        {
            "Frame stride": stride,
            "Training frames": manifest["splits"]["train"]["frames"],
            "Validation frames": manifest["splits"]["val"]["frames"],
            "Checks passed": f"{int(checks['is_pass'].sum())} of {checks.height}",
        }
    )
validation_frames = select_dataset_frames(
    frames, BASE["dataset"]["frame_stride"], BASE["dataset"]["validation_clips"]
)["val"]
annotation_path = RUN_PROJECT / "validation_annotations.json"
validation_ids = write_coco_subset(SOURCE_ROOT, validation_frames, annotation_path)
pl.from_dicts(preparation_rows)

## 3. Reference: the submitted checkpoint on the clean fold

We re-evaluate the notebook 10 checkpoint in this session with the same code and inference
settings. The experiment comparison therefore uses identical evaluation conditions.

In [ ]:
inference = BASE["inference"]
results: dict[str, dict] = {}
if SHOULD_EVALUATE:
    baseline_rows, _ = evaluate_checkpoints(
        {"best": BASELINE_WEIGHTS},
        validation_frames,
        validation_ids,
        annotation_path,
        SOURCE_ROOT,
        inference["image_sizes"][0],
        inference,
        device,
    )
    results["baseline"] = {
        **baseline_rows[0],
        "experiment": "baseline",
        "description": "Notebook 10 recipe, submitted checkpoint (epoch 1)",
        "path": str(BASELINE_WEIGHTS),
    }
    write_json(RUN_PROJECT / "baseline_evaluation.json", results["baseline"])
    display(pl.from_dicts([results["baseline"]]).drop("per_class_ap", "path"))

## 4. Training experiments

Experiments run in the order of the configuration. When Drive is configured, each run
restores its last published state first. A finished run is evaluated without retraining,
and an interrupted one resumes with its original arguments. After evaluation, the better
of `best.pt` and `last.pt` exports `yolo11s_<experiment>.csv`.

In [ ]:
drive_api, batch_folder = None, None
if drive_service is not None:
    from hm2.drive_api import DriveApi, download_current_generation, sync_checkpoint_to_drive_api

    drive_api = DriveApi(drive_service)
    batch_folder = drive_api.folder(drive_api.folder(DRIVE_ROOT, "batches"), BATCH_ID)

test_input = test_frames.rename(
    {"file_name": "relative_path", "width": "image_width", "height": "image_height"}
)


def export_submission(path: Path, name: str, settings: dict, metrics: dict) -> Path:
    predictions = predict_frames(
        YOLO(str(path)),
        test_input,
        SOURCE_ROOT,
        settings["inference"]["image_sizes"][0],
        settings["inference"],
        device,
    )
    submission = build_yolo_submission(predictions, test_frames)
    output = SUBMISSION_DIRECTORY / f"yolo11s_{name}.csv"
    incoming = output.with_suffix(".incoming")
    submission.write_csv(incoming)
    incoming.replace(output)
    with path.open("rb") as stream:
        checkpoint_digest = hashlib.file_digest(stream, "sha256").hexdigest()
    write_json(
        output.with_suffix(".metadata.json"),
        {
            "experiment": name,
            "checkpoint": str(path),
            "checkpoint_sha256": checkpoint_digest,
            "inference": settings["inference"],
            "clean_fold_map_100": metrics["map_100"],
            "rows": submission.height,
        },
    )
    return output


for name, overrides in EXPERIMENTS.items():
    settings = apply_experiment(BASE, overrides)
    run_directory = RUN_PROJECT / name
    mirror_directory = MIRROR_ROOT / name
    folder = (
        drive_api.folder(batch_folder, name) if drive_api is not None and batch_folder else None
    )
    if drive_api is not None and folder is not None:
        download_current_generation(drive_api, folder, mirror_directory)
    checkpoint_epoch = restore_run_from_mirror(mirror_directory, run_directory)
    if SHOULD_TRAIN and checkpoint_epoch != -1:
        mirror = None
        if drive_api is not None and folder is not None:

            def publish(directory: Path, folder_id: str = folder) -> None:
                publish_snapshot(directory)
                sync_checkpoint_to_drive_api(drive_api, directory, folder_id)

            mirror = CheckpointMirror(run_directory, mirror_directory, publish)
            mirror.start()
        started = time.monotonic()
        try:
            if checkpoint_epoch is None:
                YOLO(str(INITIAL_WEIGHTS)).train(
                    **training_arguments(
                        settings,
                        DATA_YAMLS[settings["dataset"]["frame_stride"]],
                        RUN_PROJECT,
                        name,
                        settings["training"]["max_epochs"],
                    )
                )
            else:
                YOLO(str(run_directory / "weights" / "last.pt")).train(resume=True)
        finally:
            if mirror is not None:
                mirror.stop()
        LOGGER.info("%s trained in %.2f h", name, (time.monotonic() - started) / 3600)
    candidates = {
        candidate: run_directory / "weights" / f"{candidate}.pt"
        for candidate in ("best", "last")
        if (run_directory / "weights" / f"{candidate}.pt").is_file()
    }
    if not (SHOULD_EVALUATE and candidates):
        continue
    rows, _ = evaluate_checkpoints(
        candidates,
        validation_frames,
        validation_ids,
        annotation_path,
        SOURCE_ROOT,
        settings["inference"]["image_sizes"][0],
        settings["inference"],
        device,
    )
    selected = max(rows, key=lambda row: row["map_100"])
    results[name] = {
        **selected,
        "experiment": name,
        "description": overrides["description"],
        "path": str(candidates[selected["candidate"]]),
        "candidates": rows,
    }
    write_json(run_directory / "evaluation.json", results[name])
    if SHOULD_EXPORT_SUBMISSIONS:
        exported = export_submission(candidates[selected["candidate"]], name, settings, selected)
        LOGGER.info("%s: exported %s", name, exported)
    if drive_api is not None and folder is not None:
        mirror_directory.mkdir(parents=True, exist_ok=True)
        shutil.copy2(run_directory / "evaluation.json", mirror_directory / "evaluation.json")
        publish_snapshot(mirror_directory)
        sync_checkpoint_to_drive_api(drive_api, mirror_directory, folder)

## 5. Comparison on the clean fold

Differences are in mAP points relative to the re-evaluated reference checkpoint. Six clips
are a small sample, so differences of about one point are within their noise. Native
validation curves show when each run peaked.

In [ ]:
if results:
    reference = results["baseline"]["map_100"] if "baseline" in results else None
    comparison = pl.from_dicts(
        [
            {
                "Experiment": record["experiment"],
                "Change": record["description"],
                "Checkpoint": record["candidate"],
                "mAP": record["map_100"],
                "Change vs reference": (record["map_100"] - reference) if reference else None,
                "AP50": record["ap50"],
                "AP75": record["ap75"],
                "AP small": record["ap_small"],
                "AP medium": record["ap_medium"],
                "AP large": record["ap_large"],
            }
            for record in results.values()
        ]
    ).sort("mAP", descending=True)
    display(comparison)
    display(
        pl.from_dicts(
            [
                {"Experiment": record["experiment"], **record["per_class_ap"]}
                for record in results.values()
            ]
        )
    )
    curves = [
        pl.read_csv(RUN_PROJECT / name / "results.csv")
        .rename(lambda column: column.strip())
        .select("epoch", pl.col("metrics/mAP50-95(B)").alias("mAP50-95"))
        .with_columns(pl.lit(name).alias("Experiment"))
        for name in EXPERIMENTS
        if (RUN_PROJECT / name / "results.csv").is_file()
    ]
    if curves:
        display(
            px.line(
                pl.concat(curves),
                x="epoch",
                y="mAP50-95",
                color="Experiment",
                markers=True,
                labels={"epoch": "Epoch", "mAP50-95": "Native validation mAP50-95"},
                title="Native per-epoch validation on the masked clean fold",
            )
        )
else:
    display(Markdown("No evaluated experiment in this session."))

## 6. Inference strategies on the best checkpoint

These settings change only inference, never the weights: a 1536 pixel input, Ultralytics
test-time augmentation with flips and scales, and a confidence threshold of 0.05 instead
of 0.001. They are applied to the best checkpoint from Sections 3 and 4.

In [ ]:
inference_rows: list[dict] = []
if SHOULD_EVALUATE and results:
    best_record = max(results.values(), key=lambda record: record["map_100"])
    best_path = Path(best_record["path"])
    inference_rows.append(
        {"Inference": "default", "Change": "Settings of notebook 10", **best_record}
    )
    for name, overrides in INFERENCE_EXPERIMENTS.items():
        settings = apply_experiment(BASE, overrides)
        rows, _ = evaluate_checkpoints(
            {"best": best_path},
            validation_frames,
            validation_ids,
            annotation_path,
            SOURCE_ROOT,
            settings["inference"]["image_sizes"][0],
            settings["inference"],
            device,
        )
        inference_rows.append({"Inference": name, "Change": overrides["description"], **rows[0]})
    write_json(
        RUN_PROJECT / "inference_evaluation.json",
        {"checkpoint": str(best_path), "rows": inference_rows},
    )
    display(Markdown(f"Checkpoint: `{best_record['experiment']}` ({best_record['candidate']})."))
    display(
        pl.from_dicts(
            [
                {
                    "Inference": row["Inference"],
                    "Change": row["Change"],
                    "mAP": row["map_100"],
                    "Change vs default": row["map_100"] - inference_rows[0]["map_100"],
                    "AP small": row["ap_small"],
                    "ms per frame": row["milliseconds_per_frame"],
                }
                for row in inference_rows
            ]
        )
    )

## 7. Final selection and submission checks

The final candidate is the combination of checkpoint and inference settings with the
highest clean-fold mAP. If it differs from an already exported file, it is exported as
`yolo11s_tuned_final.csv`. Every CSV is checked against `test_images.csv`.

In [ ]:
if SHOULD_EXPORT_SUBMISSIONS and inference_rows:
    final = max(inference_rows, key=lambda row: row["map_100"])
    final_settings = (
        BASE
        if final["Inference"] == "default"
        else apply_experiment(BASE, INFERENCE_EXPERIMENTS[final["Inference"]])
    )
    best_record = max(results.values(), key=lambda record: record["map_100"])
    if best_record["experiment"] == "baseline" and final["Inference"] == "default":
        display(Markdown("The submitted notebook 10 checkpoint remains the best candidate."))
    else:
        export_submission(Path(best_record["path"]), "tuned_final", final_settings, final)
    checks = []
    for path in sorted(SUBMISSION_DIRECTORY.glob("yolo11s_*.csv")):
        written = pl.read_csv(path, schema_overrides={"image_id": pl.String})
        metadata = json.loads(path.with_suffix(".metadata.json").read_text())
        checks.append(
            {
                "File": path.name,
                "Rows": written.height,
                "Order matches test_images.csv": written["image_id"].to_list()
                == test_frames["image_id"].to_list(),
                "Clean-fold mAP": metadata["clean_fold_map_100"],
                "MiB": path.stat().st_size / 1024**2,
            }
        )
    display(pl.from_dicts(checks))
    if drive_api is not None and batch_folder is not None:
        summary_directory = MIRROR_ROOT / "summary"
        summary_directory.mkdir(parents=True, exist_ok=True)
        for path in [
            *SUBMISSION_DIRECTORY.glob("yolo11s_*"),
            RUN_PROJECT / "baseline_evaluation.json",
            RUN_PROJECT / "inference_evaluation.json",
        ]:
            if path.is_file():
                shutil.copy2(path, summary_directory / path.name)
        publish_snapshot(summary_directory)
        sync_checkpoint_to_drive_api(
            drive_api, summary_directory, drive_api.folder(batch_folder, "summary")
        )

## Limitations

- Each experiment is one seed on six validation clips. We treat differences below about
  one mAP point as inconclusive.
- The experiments change one factor each and do not test combinations. The best single
  change is a candidate for a combined run, which this notebook does not perform.
- Test-time augmentation in Ultralytics also evaluates downscaled copies, which shrink
  small objects further. Its result is measured, not assumed.
- Choose the final leaderboard entries by clean-fold mAP, not by Public score alone.